### FIFA World Cup 2026 - Consumer Spending Accommodations and Food SARIMA Modeling
#### Forecasting restaurant and hotel consumer spending baselines for World Cup host and control (non-host) cities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Reading in data
df = pd.read_csv('../data/processed/worldcup_city_seasonal_spending.csv', index_col='Date', parse_dates=True)
df.head()

Below is a pipeline test using a host city, Atlanta, to ensure the SARIMA model is functioning correctly and for debugging purposes. This test confirms that the loop-based model produces identical results to the individually fitted Atlanta model, validating the pipeline before scaling to all 9 cities.

In [ ]:
# Testing SARIMA model on Atlanta to ensure pipline is working correctly

atl_df = df[df['cityname'] == 'Atlanta']
atl_spending = atl_df.spend_acf
print(atl_spending.head())
print()

# Running ADF test to check for stationarity
result = adfuller(atl_spending)
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')

# auto_arima model to find best parameters for SARIMA model
auto_model = auto_arima(
    atl_spending,
    seasonal=True,
    m=12,
    stepwise=True,
    suppress_warnings=True,
    information_criterion='aic'
)

# Fit SARIMA model with best parameters found by auto_arima
sarima_model = SARIMAX(
    atl_spending,
    order=(1, 0 ,0),
    seasonal_order=(0, 1, 1, 12))
model_fit = sarima_model.fit(disp=False) 

In [ ]:
# Generate forecasts for the next 12 months
forecast = model_fit.get_forecast(steps=12)
forecast_mean = forecast.predicted_mean
forecast_ci = forecast.conf_int()
print(forecast_mean)

In [ ]:
# Plotting Atlanta's forecasted spending trends

plt.figure(figsize=(12, 6))
plt.plot(atl_spending, label='Historical', color='blue')
plt.plot(forecast_mean, label='Forecast', color='green')
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 alpha=0.3,
                 label='Confidence Interval')
plt.title("Atlanta Spending Forecast")
plt.xlabel("Year")
plt.ylabel("% Change in Restaurant & Accommodation Spending")
plt.legend()
plt.savefig('../outputs/forecasting_figures/Atlanta_forecast.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

In [ ]:
# Predicting spending for all cities using SARIMA model

# Overwriting df so all cities used, not only Atlanta  
df = pd.read_csv('../data/processed/worldcup_city_seasonal_spending.csv', index_col='Date', parse_dates=True)

target_cities = ['Los Angeles', 'Denver', 'Dallas', 'Chicago', 'Boston', 'Atlanta', 'Kansas City', 'Austin', 'Charlotte']
host_cities = ['Los Angeles', 'Atlanta', 'Dallas', 'Boston', 'Kansas City']
nonhost_cities = ['Denver', 'Chicago', 'Austin', 'Charlotte']

forecasts = {}

for city in target_cities:
    # Filter to city
    city_df = df[df['cityname'] == city]

    # Conduct ADF test to check for stationarity
    result = adfuller(city_df.spend_acf)
    print(f'ADF Statistic for {city}: {result[0]:.4f}')
    print(f'p-value for {city}: {result[1]:.4f}')

    # Run auto_arima to find best parameters for SARIMA model
    auto_model = auto_arima(
        city_df.spend_acf,
        seasonal=True,
        m=12,
        stepwise=True,
        suppress_warnings=True,
        information_criterion='aic'
    )
    
    # Fit SARIMA model with best parameters found by auto_arima
    model = SARIMAX(
        city_df.spend_acf,
        order=auto_model.order,
        seasonal_order=auto_model.seasonal_order
    )
    model_fit = model.fit(disp=False)
    
    # Generate forecast
    forecast = model_fit.get_forecast(steps=12)
    forecast_mean = forecast.predicted_mean
    forecast_ci = forecast.conf_int()

    # Store in forecasts dictionary
    forecasts[city] = {
        'mean': forecast_mean,
        'ci': forecast_ci,
        'historical': city_df['spend_acf']
    }

    print(f'{city} done - order: {auto_model.order}, seasonal: {auto_model.seasonal_order}')


In [ ]:
# Plotting the forecast for each city (baseline diff)

for city in target_cities:
    plt.figure(figsize=(12, 6))
    plt.plot(forecasts[city]['historical'], label='Historical', color='blue')
    plt.plot(forecasts[city]['mean'], label='Forecast', color='green')
    plt.fill_between(forecasts[city]['ci'].index,
                     forecasts[city]['ci'].iloc[:, 0],
                     forecasts[city]['ci'].iloc[:, 1], alpha=0.5)
    plt.title(f'Forecast for {city}')
    plt.xlabel('Year')
    plt.ylabel("% Change in Restaurant & Accommodation Spending")
    plt.legend()
    plt.savefig(f'../outputs/forecasting_figures/{city}_forecast.png', bbox_inches='tight', dpi=150)
    plt.show()
    plt.close()

We can see Atlanta had the same graph, p-value, and ADF statistics so the SARIMA ran successfully.